# 01 — Exploratory Data Analysis (EDA)

Analyze the synthetic instruction-following dataset:
- Domain distribution
- Instruction and response length statistics
- Sample inspection

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Generate dataset (or load from file if already generated)
from llm_finetuning.data.synthetic_generator import SyntheticGeneratorConfig, generate_dataset

cfg = SyntheticGeneratorConfig(num_samples=10000, seed=42)
samples = generate_dataset(cfg)
print(f'Total samples: {len(samples)}')

In [ ]:
# Build a DataFrame for analysis
rows = []
for s in samples:
    messages = s['messages']
    user_msg = next((m['content'] for m in messages if m['role'] == 'user'), '')
    asst_msg = next((m['content'] for m in messages if m['role'] == 'assistant'), '')
    rows.append({
        'domain': s['domain'],
        'instruction_len': len(user_msg),
        'response_len': len(asst_msg),
        'instruction_words': len(user_msg.split()),
        'response_words': len(asst_msg.split()),
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
# Domain distribution
domain_counts = df['domain'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
domain_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Sample Count per Domain')
axes[0].set_xlabel('Domain')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

domain_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Domain Distribution (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()
print(domain_counts)

In [ ]:
# Length statistics
print('=== Instruction length (chars) ===')
print(df['instruction_len'].describe())
print()
print('=== Response length (chars) ===')
print(df['response_len'].describe())

In [ ]:
# Length distribution plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df['instruction_words'].hist(bins=30, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Instruction Word Count')

df['response_words'].hist(bins=30, ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Response Word Count')

df.groupby('domain')['instruction_words'].mean().plot(kind='bar', ax=axes[1, 0], color='steelblue')
axes[1, 0].set_title('Avg Instruction Words per Domain')
axes[1, 0].tick_params(axis='x', rotation=45)

df.groupby('domain')['response_words'].mean().plot(kind='bar', ax=axes[1, 1], color='coral')
axes[1, 1].set_title('Avg Response Words per Domain')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Inspect random samples per domain
for domain in df['domain'].unique():
    sample = df[df['domain'] == domain].sample(1, random_state=42)
    idx = sample.index[0]
    s = samples[idx]
    print(f'\n=== Domain: {domain} ===')
    for msg in s['messages']:
        print(f"[{msg['role'].upper()}]: {msg['content'][:200]}..." if len(msg['content']) > 200 else f"[{msg['role'].upper()}]: {msg['content']}")